# Týždeň 2: Aký rýchly je náš web?
### Popisná štatistika: priemer, medián, percentily, rozptyl

**Tiket SHOP-104** (Martin Novák, CTO):
> *Zákazníci sa sťažujú, že web je pomalý. Grafana ukazuje priemernú odozvu 160 ms, to je predsa v pohode. Over to.*

**Ako pracovať s notebookom:** bunky postupne spúšťaj (Shift+Enter). Kde je `...`, doplníš kód počas prednášky, keď ťa k tomu vyzve vyučujúci (🖐).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (10, 4)
DATA = "https://raw.githubusercontent.com/peternather/pas-student/main/data/shopit/"

## Časť 1: Prvý pohľad na dáta

Access log zaznamenáva každý HTTP request: kedy prišiel, kto ho poslal, na ktorý endpoint, ktorý server ho obslúžil a ako dlho to trvalo. Máme náhodnú vzorku 150 000 requestov za 12 týždňov.

In [ ]:
requests = pd.read_csv(DATA + "requests.csv", parse_dates=["ts"])
print(requests.shape)
requests.head()

In [ ]:
# Kto posiela requesty a ako dopadli?
print(requests["client"].value_counts(), "\n")
print(requests["status"].value_counts())

In [ ]:
# Chýbajúce hodnoty v každom stĺpci
requests.isna().sum()

**Skôr než niečo spočítame, rozhodnime, čo vlastne meriame.**

- **Boty** (`client == "bot"`) sú crawlery vyhľadávačov. Zákazníka nezaujíma, ako rýchlo dostal odpoveď Googlebot. Otázka znie „aký rýchly je web *pre zákazníkov*“, preto boty vyhodíme.
- **Prázdna latencia** znamená timeout (status 504): load balancer request po 10 s ukončil. Nie je to „chýbajúca hodnota bez významu“, ale *najhorší možný zážitok*. Zatiaľ ich odložíme bokom a v časti 5 sa k nim vrátime.

In [ ]:
humans = requests[requests["client"] == "browser"]
latency = humans["latency_ms"].dropna()
print(f"requesty od ľudí: {len(humans)}, timeouty: {humans['latency_ms'].isna().sum()}")

## Časť 2: Priemer a čo skrýva

In [ ]:
latency.mean()

Grafana nepreháňa: priemer je naozaj ~160 ms. Je to ale **typický** request? Pozrime sa na celé rozdelenie hodnôt v histograme.

In [ ]:
plt.hist(latency, bins=100, range=(0, 1000))
plt.axvline(latency.mean(), color="black", linestyle="--", label="priemer")
plt.xlabel("latencia [ms]")
plt.ylabel("počet requestov")
plt.legend()
plt.show()

Rozdelenie je **zošikmené doprava**: väčšina requestov je rýchla, ale niektoré trvajú veľmi dlho (dlhý pravý chvost). Tých pár pomalých requestov ťahá priemer nahor. Priemer preto nepopisuje „typický“ request.

### 🖐 Úloha 1
Spočítaj **medián** (polovica requestov je rýchlejšia), **95. a 99. percentil** a aký podiel requestov je pomalší než priemer.
Pomôcka: `Series.median()`, `Series.quantile(0.95)`, porovnanie `latency > latency.mean()` dáva True/False a priemer z True/False je podiel.

In [ ]:
median = ...
p95 = ...
p99 = ...
share_slower_than_mean = ...
print(f"medián {median:.0f} ms, p95 {p95:.0f} ms, p99 {p99:.0f} ms")
print(f"pomalších než priemer: {share_slower_than_mean:.0%}")

**Percentil** (kvantil): p95 = 378 ms znamená, že 95 % requestov je rýchlejších a **každý 20. request je pomalší než 378 ms**. Medián je 50. percentil.

Všetky percentily naraz ukazuje **empirická distribučná funkcia** (ECDF): pre každú latenciu *x* udáva, aký podiel requestov je rýchlejší než *x*.

In [ ]:
sorted_latency = np.sort(latency)
share = np.arange(1, len(sorted_latency) + 1) / len(sorted_latency)
plt.plot(sorted_latency, share)
for q in [0.5, 0.95, 0.99]:
    plt.axhline(q, color="grey", linewidth=0.8)
    plt.axvline(latency.quantile(q), color="grey", linewidth=0.8)
plt.xlim(0, 1000)
plt.xlabel("latencia [ms]")
plt.ylabel("podiel rýchlejších requestov")
plt.show()

### Prečo nás zaujíma p95, keď je to „iba“ 5 % requestov?

Jedno zobrazenie stránky ShopIT spustí približne **10 requestov** (HTML, API pre odporúčania, košík…). Ak sú requesty nezávislé, pravdepodobnosť, že **aspoň jeden** z 10 je pomalší než p95, je

$$P(\text{aspoň jeden pomalý}) = 1 - P(\text{žiadny pomalý}) = 1 - 0{,}95^{10} \approx 0{,}40.$$

Overme to aj simuláciou: 100 000-krát náhodne vyberieme 10 requestov z našich dát.

In [ ]:
print("vzorec:", 1 - 0.95 ** 10)

rng = np.random.default_rng(1)
page_views = rng.choice(latency.to_numpy(), size=(100_000, 10))
print("simulácia:", (page_views.max(axis=1) > p95).mean())

**Približne 40 % zobrazení stránky** obsahuje aspoň jeden request pomalší než 378 ms. Preto firmy sledujú p95 a p99 (napríklad v SLO: „p95 < 300 ms“) a nie priemer.

## Časť 3: Rozptyl – ako veľmi sa hodnoty líšia

- **Smerodajná odchýlka** (std): typická vzdialenosť od priemeru, $s = \sqrt{\frac{1}{n-1}\sum (x_i - \bar x)^2}$.
- **Medzikvartilové rozpätie** (IQR) = p75 − p25: šírka „strednej polovice“ dát.

Porovnajme ich na malej vzorke 20 requestov a potom pridajme jediný request, ktorý trval 9,9 s.

In [ ]:
def summary(x):
    x = pd.Series(x)
    return pd.Series({"mean": x.mean(), "median": x.median(), "std": x.std(),
                      "IQR": x.quantile(0.75) - x.quantile(0.25)}).round(1)

sample = latency.sample(20, random_state=7)
with_outlier = pd.concat([sample, pd.Series([9900.0])])
pd.DataFrame({"20 requestov": summary(sample), "+ jeden request 9,9 s": summary(with_outlier)})

Jediná extrémna hodnota posunula priemer a smerodajnú odchýlku mnohonásobne, medián a IQR sa takmer nezmenili.

| Robustné (odolné voči extrémom) | Citlivé na extrémy |
|---|---|
| medián, percentily, IQR | priemer, smerodajná odchýlka, min, max |

Citlivé neznamená zlé. Priemer potrebujeme napríklad na výpočet celkovej záťaže (súčet = priemer × počet). Pri opise „typickej“ hodnoty zošikmených dát však uprednostni robustné miery.

In [ ]:
latency.describe(percentiles=[0.25, 0.5, 0.75, 0.95, 0.99]).round(1)

## Časť 4: Porovnanie skupín – kde je web pomalý?

**Boxplot** zhrnie rozdelenie do piatich čísel: krabica ide od p25 po p75, čiara v nej je medián, „fúzy“ siahajú po najvzdialenejšiu hodnotu do 1,5 × IQR od krabice a bodky za nimi sú odľahlé hodnoty.

In [ ]:
order = humans.groupby("endpoint")["latency_ms"].median().sort_values().index
data = [humans.loc[humans["endpoint"] == e, "latency_ms"].dropna() for e in order]
plt.boxplot(data, tick_labels=order, showfliers=False)
plt.ylabel("latencia [ms]")
plt.title("Latencia podľa endpointu (bez odľahlých hodnôt)")
plt.show()

### 🖐 Úloha 2
Vytvor tabuľku: pre každý endpoint počet requestov, priemer, medián, p95 a p99 latencie, zoradenú podľa p95 zostupne.
Pomôcka: `groupby("endpoint")["latency_ms"].agg(...)`; percentil v `agg` zapíšeš ako `lambda s: s.quantile(0.95)`.

In [ ]:
by_endpoint = ...
by_endpoint

## Časť 5: Pasce – boty a timeouty

Ako by sa zmenili čísla, keby sme boty nevyhodili, alebo keby sme timeouty započítali ako 10 s (čo je dolná hranica toho, ako dlho zákazník čakal)?

In [ ]:
def latency_stats(x):
    return pd.Series({"mean": x.mean(), "median": x.median(),
                      "p95": x.quantile(0.95), "p99": x.quantile(0.99)}).round(0)

pd.DataFrame({
    "ľudia, bez timeoutov": latency_stats(latency),
    "ľudia + boty": latency_stats(requests["latency_ms"].dropna()),
    "ľudia, timeout = 10 s": latency_stats(humans["latency_ms"].fillna(10_000)),
})

- **Boty** sú rýchle (dostávajú stránky z cache), takže ich započítanie robí web zdanlivo rýchlejším.
- **Timeouty** sú iba 0,5 % requestov, ale ak ich započítame, priemer stúpne o ~50 ms a p99 o ~200 ms. Keď ich ticho vyhodíme (`dropna()`), zakryjeme najhoršie zážitky zákazníkov.

Ani jedna verzia nie je „tá správna“. Správne je **povedať, čo si spravil**: „bez botov; timeouty (0,5 %) sú vylúčené, s nimi by bolo p99 = 889 ms“.

### Pohľad na budúce týždne: logaritmická os
Na logaritmickej osi vyzerá rozdelenie latencie takmer symetricky. K tomu, prečo to tak je, sa vrátime vo 4. týždni (log-normálne rozdelenie).

In [ ]:
plt.hist(latency, bins=np.logspace(1, 4, 80))
plt.xscale("log")
plt.xlabel("latencia [ms], logaritmická os")
plt.ylabel("počet requestov")
plt.show()

## Kuchárka: popisná štatistika jednej číselnej premennej

1. **Čo meriam a koho?** Vyfiltruj, čo do otázky nepatrí (boty), a zapíš si to.
2. **Chýbajúce hodnoty:** koľko ich je a *prečo* chýbajú? Neznamenajú niečo (timeout)?
3. **Histogram** (pri dlhom chvoste aj na log osi): je rozdelenie symetrické, alebo zošikmené?
4. **Stred:** pri zošikmených dátach medián, pri symetrických aj priemer.
5. **Rozptyl:** IQR a percentily (p90, p95, p99); smerodajnú odchýlku iba pri symetrických dátach.
6. **Skupiny:** boxplot a tabuľka cez `groupby`.
7. **Záver:** čísla s jednotkami, čo si vylúčil a prečo, a odpoveď na otázku z tiketu.

## Odpoveď na tiket SHOP-104
Napíš Martinovi odpoveď v 3–5 vetách: je web pomalý? Ktoré číslo má sledovať namiesto priemeru? Kde je problém najväčší?